# ColabDFT MO — 分子軌道（HOMO・LUMO）の表示

入力した構造のまま、[GPU4PySCF](https://github.com/pyscf/gpu4pyscf) で DFT の一点計算を行い、**HOMO・LUMO などの分子軌道を等値面で 3D 表示** するノートブックです。構造の最適化はしません。コードを書く必要はありません。UMA は使わないので、Hugging Face のトークンはいりません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、汎関数、基底関数、表示する軌道を指定する
3. **ランタイム → すべて実行** を押す
4. **2. ファイル選択と事前確認** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
5. 下のセルで軌道の 3D 表示とエネルギーを確認し（準位図をクリックすると、ほかの軌道も見られます）、最後のセルで結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。計算方法はフォームの値を使い、電荷と多重度もフォームの値を使います。xyz に複数の構造が入っている場合は先頭だけを計算します。

2回目以降は手順2〜5を繰り返すだけです（インストールは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_spe.xyz` | 計算した構造。コメント行にエネルギーと計算条件 |
| `<出力名>_mo<番号>_<軌道>.cube` | 格子上の軌道の値（Bohr^-3/2）。例：`_mo5_homo.cube`、`_mo6_lumo.cube`、`_mo4_homom1.cube`（HOMO-1）。開殻では `_alpha` / `_beta` が付く。GaussView などで開ける |
| `<出力名>_mo_energies.csv` | 全軌道のエネルギー（Hartree・eV）と占有数 |
| `<出力名>_dft.csv` | DFT 一点計算のエネルギー |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、計算結果、HOMO・LUMO のエネルギーとギャップ |

## 注意事項

- 等値面は **青が正、赤が負** です。軌道の符号（位相）には意味がなく、計算ごとに青と赤が入れ替わることがあります。
- 軌道は Kohn–Sham 軌道です。軌道のエネルギーや HOMO-LUMO ギャップは汎関数で大きく変わります（実験のイオン化エネルギーや吸収の波長とは一致しません）。
- 表示する軌道は「HOMO, LUMO」のようにカンマで区切って指定します。HOMO-1、LUMO+1 のような書き方や、軌道の番号（エネルギーの低い順に 1 から）も使えます。
- 開殻（多重度2以上）では、α と β の軌道をそれぞれ表示します。α の HOMO が SOMO にあたります。
- 「7. 軌道の表示」では、エネルギー準位図の線をクリックすると、その軌道を計算して表示します。HOMO の下と LUMO の上の 8 個ずつが並びます（内殻の軌道は出しません）。フォームで指定した軌道はノートに保存されますが、それ以外の軌道はランタイムにつながっている間だけ計算できます。
- ZIP に入る cube は、フォームで指定した軌道だけです。準位図で見つけた軌道を保存したいときは、フォームに足して実行し直してください。
- 3D 表示はノートが重くならないよう格子を間引いています。ZIP の cube は間引いていません。
- ECP を使う元素（Rb 以降）は内殻の電子を ECP に置き換えるので、その分だけ軌道の番号が小さくなります（HOMO・LUMO の指定はそのまま使えます）。
- DFT は GPU4PySCF（密度フィッティング）で計算します。T4 は倍精度の計算が遅いため、原子数が多いときや def2-TZVPD では時間がかかります。GPU が使えないときは CPU の PySCF で計算します。
- 閉殻（多重度1）は RKS、開殻は UKS で計算します。
- 扱える元素は H〜Rn（原子番号1〜86）です。周期系（結晶・表面）には対応していません。
- UMA で構造を最適化してから一点計算するときは [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt.ipynb) を使ってください。
- 静電ポテンシャルを見るときは [ColabDFT ESP](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_ESP.ipynb) を使ってください。
- エネルギーだけでよいときは [ColabDFT_SPE](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_SPE.ipynb) のほうが速く終わります。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）
- GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### DFT 計算（gpu4pyscf）
#@markdown ωB97M-V/def2-TZVPD が UMA（OMol25）の学習データと同じ計算レベルです。def2-TZVPD は T4 では1点に数十分かかることがあります。
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V（OMol25と同じ）", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]

#@markdown ### 軌道の表示
#@markdown 表示する軌道をカンマで区切って書きます。例：`HOMO, LUMO`、`HOMO-1, HOMO, LUMO, LUMO+1`。軌道の番号（1始まり）でも指定できます。
orbitals = "HOMO, LUMO"  #@param {type:"string"}
#@markdown 等値面の値 ±（Bohr^-3/2）。0.02 が GaussView の既定値と同じです。大きくすると面が小さくなります。
isovalue = 0.02  #@param {type:"number"}
#@markdown 格子の細かさ。細かいほどきれいですが、時間とファイルサイズが増えます。
grid_quality = "標準"  #@param ["粗い", "標準", "細かい"]

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択をここで済ませます。このあとのインストールと計算は放置できます。

import os
import subprocess
import sys

COLABUMA_REF = "main"  # 共通コード（colabuma パッケージ）を取ってくる GitHub のブランチ


def _use_colabuma(ref):
    # ノートと同じフォルダに colabuma があればそれを、なければ GitHub から取ってきて使う
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"ColabChem の共通コードを GitHub から取得できませんでした（ブランチ {ref}）。\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, mo, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

structure.check_multiplicity(multiplicity)
functional, basis = dft.parse_method(dft_functional, dft_basis)
orbital_specs = mo.check_form(orbitals, isovalue, grid_quality)
job = dft.Job(structure.out_prefix(output_name), charge, multiplicity, functional, basis)
env.check_gpu()
upload_name, upload_data = structure.upload()
ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール） { display-mode: "form" }
DEVICE = env.setup_dft()

In [ ]:
#@title 4. 構造の準備 { display-mode: "form" }
atoms, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity, uma=False)
structure.show_numbered(atoms)
ok(f"{atoms.get_chemical_formula()}（{len(atoms)} 原子）の構造を用意しました。")

In [ ]:
#@title 5. DFT 計算 { display-mode: "form" }
dft_res = dft.run([("入力構造", atoms)], charge, multiplicity, functional, basis, keep_orbitals=True)
files, dft_log = dft.report(dft_res, None, geometry="入力の構造")
_p = dft_res["points"][0]
files["spe.xyz"] = structure.xyz_block(atoms.get_chemical_symbols(), atoms.positions,
                                       f"energy_Ha={_p['energy']:.10f} converged={_p['converged']} "
                                       f"method={functional}/{basis} charge={charge} multiplicity={multiplicity}")

In [ ]:
#@title 6. 軌道の計算 { display-mode: "form" }
mo_res = mo.compute(dft_res["points"][0], atoms, orbital_specs, isovalue, grid_quality)

In [ ]:
#@title 7. 軌道の表示 { display-mode: "form" }
#@markdown 左のエネルギー準位図の線をクリックすると、その軌道を右に表示します（HOMO の下と LUMO の上の 8 個ずつ）。青が正、赤が負の等値面で、マウスで回転・拡大できます。
_f, mo_log = mo.report(mo_res)
files.update(_f)

In [ ]:
#@title 8. 保存 { display-mode: "form" }
files["log.txt"] = output.log_text("ColabDFT MO 計算ログ", source_desc, job, [dft_log, mo_log])
zip_path = output.save_zip(job.out_prefix, files)